# v3b — Perceiver + GPT-2 + Cross-Attention Injection

| Field | Value |
|---|---|
| Framework | PyTorch |
| Dataset | Flickr30k (31,783 images, 5 captions each) |
| Image Encoder | ConvNeXt-Tiny multi-scale → LMDB (same as v3a) |
| Visual Compressor | Perceiver: (B, 64, 768) → (B, 16, 768) |
| Decoder | GPT-2 (pretrained) + CrossAttentionBlock injected at all 12 layers |
| CrossAttention | query = text hidden states, key/value = Perceiver latents (768-dim, 12 heads) |
| Loss | CrossEntropyLoss(ignore_index=50256) — padding masked, fixed from v3a |
| Inference | Top-k=20, temperature=0.7, retry logic for degenerate outputs |
| BLEU-4 | 0.0656 |
| Status | Superseded by v3c (COCO 2017 fine-tuning → BLEU 0.1341) |

## Overview

v3a proved that ConvNeXt multi-scale features + a Transformer decoder can generate fluent captions (BLEU 0.0674). But the custom decoder was trained from scratch — it had to simultaneously learn language and visual grounding. v3b splits that burden:

- **GPT-2 handles language.** Its 12 pretrained transformer layers already know English grammar, common phrases, and world knowledge. We freeze most of its parameters and inject image information surgically.
- **Perceiver handles visual compression.** Instead of feeding all 64 image tokens into cross-attention at every GPT-2 layer, the Perceiver distills them into 16 latent vectors — reducing attention complexity and forcing the model to extract the most task-relevant visual information.
- **CrossAttentionBlock handles injection.** A single `MultiheadAttention` block (query = text hidden state, key/value = Perceiver latents) is inserted after the self-attention step of every GPT-2 layer. This gives the language model a persistent image signal at every depth.

**Training was done in 4 phases**, progressively unfreezing components:
- **Phase 2a:** Only Perceiver trained; GPT-2 fully frozen; prepend image tokens to caption sequence
- **Phase 2b:** Perceiver + GPT-2 jointly fine-tuned; still prepend approach; DataParallel
- **Phase 2c:** Architecture switched to CrossAttention injection; only CrossAttention blocks trained; Perceiver and GPT-2 frozen
- **Phase 2d:** All components fine-tuned jointly

Final BLEU-4: **0.0656** — marginally below v3a (0.0674) on Flickr30k alone. The architecture reaches its potential only after COCO fine-tuning in v3c (BLEU 0.1341).

## 1. Setup

Dataset, LMDB, tokenizer, and DataLoader are identical to v3a. Setup is reproduced here for standalone executability.

In [ ]:
import os
import re
import csv
import random
import numpy as np
import lmdb

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from transformers import GPT2Tokenizer, GPT2LMHeadModel

import nltk
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from nltk.corpus import words as nltk_words
nltk.download('punkt', quiet=True)
nltk.download('words', quiet=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

In [ ]:
# ── Tokenizer ──────────────────────────────────────────────────────────────────
tokenizer = GPT2Tokenizer.from_pretrained('gpt2')
tokenizer.pad_token = tokenizer.eos_token
PAD_ID = BOS_ID = EOS_ID = tokenizer.eos_token_id  # all 50256

# ── Flickr30k captions ────────────────────────────────────────────────────────
def load_captions(filepath):
    captions_dict = {}
    with open(filepath, 'r', encoding='utf-8') as f:
        reader = csv.DictReader(f, delimiter='|')
        for row in reader:
            name = row['image_name'].strip()
            num  = row[' comment_number'].strip()
            cap  = row[' comment'].strip()
            captions_dict.setdefault(name, {})[num] = cap
    return captions_dict

flickr_captions_dict = load_captions('/kaggle/input/flickr30k/captions.txt')
all_keys = list(flickr_captions_dict.keys())

random.seed(42)
random.shuffle(all_keys)
split      = int(0.9 * len(all_keys))
train_keys = all_keys[:split]
val_keys   = all_keys[split:]
print(f'Train: {len(train_keys)} | Val: {len(val_keys)}')

# ── LMDB (ConvNeXt embeddings, float16, shape 64×768) ─────────────────────────
N_SHARDS = 6
envs = [
    lmdb.open(f'/kaggle/input/datasets/huzefamerchant/lmdb-part-{i}',
              readonly=True, lock=False, readahead=False)
    for i in range(1, N_SHARDS + 1)
]

def get_embedding(key):
    key_bytes = key.encode()
    for env in envs:
        with env.begin() as txn:
            value = txn.get(key_bytes)
            if value is not None:
                return np.frombuffer(value, dtype=np.float16).reshape(64, 768)
    return None

# ── Dataset & DataLoader ───────────────────────────────────────────────────────
MAX_LEN = 64

class FlickrDataset(Dataset):
    def __init__(self, keys, captions_dict, max_len=MAX_LEN):
        self.samples = [(k, cap) for k in keys for cap in captions_dict[k].values()]
        self.max_len = max_len

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        key, caption = self.samples[idx]
        image_tensor = torch.from_numpy(get_embedding(key).copy()).float()
        token_ids = [BOS_ID] + tokenizer.encode(caption) + [EOS_ID]
        token_ids = token_ids[:self.max_len]
        token_ids += [PAD_ID] * (self.max_len - len(token_ids))
        return image_tensor, torch.tensor(token_ids, dtype=torch.long)

train_dataset    = FlickrDataset(train_keys, flickr_captions_dict)
val_dataset      = FlickrDataset(val_keys,   flickr_captions_dict)
train_dataloader = DataLoader(train_dataset, batch_size=32, shuffle=True,  num_workers=2, pin_memory=True)
val_dataloader   = DataLoader(val_dataset,   batch_size=32, shuffle=False, num_workers=2, pin_memory=True)
print(f'Train samples: {len(train_dataset)} | Val samples: {len(val_dataset)}')

## 2. Architecture Overview

```
LMDB embedding (64, 768)
        │
   Perceiver
  (16 latents)         ← compresses 64 image tokens → 16 task-relevant latent vectors
        │
        ▼
 perciever_image (16, 768)
        │
        ├────────────────────────────────────────────┐
        │                                            │
  GPT-2 Block ×12:                          CrossAttentionBlock ×12
    ln_1 → self-attn                      (query=text, key/value=perceiver_image)
        ↓                                            │
    + CrossAttention ◄──────────────────────────────┘
        ↓
    ln_2 → mlp
        │
   lm_head (50257)
```

The critical design choice: **image context is injected at every one of GPT-2's 12 layers**, not just at the input. This forces every level of language processing — from token n-gram patterns in early layers to high-level semantic composition in later layers — to be conditioned on the image.

## 3. Perceiver — Visual Compression

The Perceiver takes the 64 ConvNeXt spatial tokens `(B, 64, 768)` and compresses them to 16 learned latent vectors `(B, 16, 768)`.

**Why not feed all 64 tokens directly into cross-attention?**
Because cross-attention is applied at every one of GPT-2's 12 layers, 12× per forward pass. With 64 keys, that's 12 × 64 = 768 key-value lookups per sequence step. With 16 latents, it's 12 × 16 = 192 — 4× cheaper, and the Perceiver is trained to pack the most useful information into those 16 slots.

**How it works:**
1. Linear(768→768) projects image tokens
2. `num_latents=16` learnable parameter vectors are broadcast to batch size
3. Cross-attention: latents query the image tokens → latents absorb visual information
4. 4× self-attention (`TransformerEncoderLayer`) refines the latents among themselves

In [ ]:
class Perceiver(nn.Module):
    def __init__(self, d_model=768, num_latents=16, num_layers=4, nhead=8):
        super().__init__()
        self.linear  = nn.Linear(768, 768)
        self.latents = nn.Parameter(torch.randn(num_latents, d_model))  # learned queries
        self.cross_attn = nn.MultiheadAttention(d_model, nhead, batch_first=True)
        self.self_attn  = nn.ModuleList([
            nn.TransformerEncoderLayer(d_model, nhead, activation='gelu', batch_first=True)
            for _ in range(num_layers)
        ])

    def forward(self, x):                              # x: (B, 64, 768)
        x = self.linear(x)                             # (B, 64, 768)
        B = x.size(0)
        latents = self.latents.unsqueeze(0).repeat(B, 1, 1)   # (B, 16, 768)
        latents, _ = self.cross_attn(query=latents, key=x, value=x)  # latents absorb image
        for layer in self.self_attn:
            latents = layer(latents)                   # latents refine themselves
        return latents                                 # (B, 16, 768)

## 4. CrossAttentionBlock

A minimal cross-attention module inserted into each GPT-2 transformer block. It follows the pre-norm convention (LayerNorm before attention) and uses a residual connection.

- **query:** the current text hidden state `x` (shape `(B, T, 768)`) — normalized first
- **key / value:** the 16 Perceiver latents `(B, 16, 768)`
- **output:** text hidden state updated with visual context, same shape `(B, T, 768)`

Note the dimension mismatch: GPT-2 hidden states are 768-dim and Perceiver latents are also 768-dim, so no projection is needed. The 12 attention heads match GPT-2's native head count.

In [ ]:
class CrossAttentionBlock(nn.Module):
    def __init__(self, d_model=768, nhead=12):
        super().__init__()
        self.layernorm  = nn.LayerNorm(d_model)
        self.cross_attn = nn.MultiheadAttention(d_model, nhead, batch_first=True)

    def forward(self, perciever_image, x):    # perciever_image: (B,16,768)  x: (B,T,768)
        x_normed        = self.layernorm(x)
        cross, _        = self.cross_attn(query=x_normed,
                                          key=perciever_image,
                                          value=perciever_image)
        return x + cross                      # residual connection → (B, T, 768)

## 5. GPT-2 with Cross-Attention Injection

`GPT2withCrossAttention` surgically rewires GPT-2's forward pass. Instead of calling `gpt2.forward()` as a black box, we replicate the computation layer-by-layer and insert a `CrossAttentionBlock` between self-attention and the feed-forward network at each of the 12 blocks.

**Replicated components (references to GPT-2 weights, not copies):**
- `wte` — token embedding
- `wpe` — positional embedding
- `h` — 12 transformer blocks (each has `ln_1`, `attn`, `ln_2`, `mlp`)
- `ln_f` — final layer norm
- `lm_head` — linear projection to vocabulary (50257)

**Modified forward loop (per block):**
```
x = x + attn(ln_1(x))          ← original GPT-2 self-attention
x = CrossAttentionBlock(image, x)  ← our injection
x = x + mlp(ln_2(x))           ← original GPT-2 feed-forward
```

In [ ]:
class GPT2withCrossAttention(nn.Module):
    def __init__(self):
        super().__init__()
        gpt2 = GPT2LMHeadModel.from_pretrained('gpt2')
        gpt2.gradient_checkpointing_enable()     # must be called before extracting references — the extracted components are the same Python objects

        # References to pretrained GPT-2 components (not copies)
        self.gpt2_blocks = gpt2.transformer.h    # 12 transformer blocks
        self.wte         = gpt2.transformer.wte  # token embedding
        self.wpe         = gpt2.transformer.wpe  # positional embedding
        self.ln_f        = gpt2.transformer.ln_f # final LayerNorm
        self.lm_head     = gpt2.lm_head          # vocab projection (768 → 50257)

        # One CrossAttentionBlock per GPT-2 layer — trained from scratch
        self.cross_attn = nn.ModuleList([
            CrossAttentionBlock(d_model=768, nhead=12) for _ in range(12)
        ])

    def forward(self, perciever_image, caption_ids):   # perciever_image: (B,16,768)
        B, T = caption_ids.shape

        # Token + positional embeddings (GPT-2 style)
        token_emb = self.wte(caption_ids)                           # (B, T, 768)
        pos_ids   = torch.arange(T, device=caption_ids.device)
        pos_emb   = self.wpe(pos_ids)                               # (T, 768)
        x         = token_emb + pos_emb                             # (B, T, 768)

        # 12-layer loop: self-attention → cross-attention → feed-forward
        for block, cross in zip(self.gpt2_blocks, self.cross_attn):
            x = x + block.attn(block.ln_1(x))[0]   # GPT-2 self-attention
            x = cross(perciever_image, x)            # image injection
            x = x + block.mlp(block.ln_2(x))        # GPT-2 feed-forward

        x   = self.ln_f(x)                           # (B, T, 768)
        out = self.lm_head(x)                        # (B, T, 50257)
        return out

## 6. Full Decoder — PercieverGptDecoder

The top-level module composes Perceiver and GPT2withCrossAttention. The forward pass:
1. Perceiver compresses `(B, 64, 768)` → `(B, 16, 768)` latents
2. GPT-2 generates caption logits conditioned on the latents at every layer
3. Output is transposed to `(B, 50257, T)` for `nn.CrossEntropyLoss`

In [ ]:
class PercieverGptDecoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.perciever = Perceiver(d_model=768, num_latents=16, num_layers=4, nhead=8)
        self.gpt2      = GPT2withCrossAttention()

    def forward(self, image_embedding, caption_ids):  # image_embedding: (B,64,768)
        perciever_image = self.perciever(image_embedding)        # (B, 16, 768)
        logits          = self.gpt2(perciever_image, caption_ids)  # (B, T, 50257)
        return logits.transpose(1, 2)                            # (B, 50257, T)


# Shape sanity check
model = PercieverGptDecoder().to(device)
dummy_img = torch.randn(2, 64, 768).to(device)
dummy_cap = torch.randint(0, 50257, (2, 32)).to(device)
out = model(dummy_img, dummy_cap)
print(f'Output shape: {out.shape}')  # (2, 50257, 32)
del model, dummy_img, dummy_cap, out

## 7. Training Evolution — Four Phases

The model was not trained end-to-end from the start. Unfreezing was staged to avoid destroying GPT-2's pretrained weights:

| Phase | What was trained | GPT-2 state | Architecture | Epochs | LR | BLEU-4 |
|---|---|---|---|---|---|---|
| **2a** | Perceiver only | Frozen | Prepend: image latents concatenated before token sequence | 25 | 1e-5 | — |
| **2b** | Perceiver + GPT-2 jointly | Unfrozen | Prepend (same) | 25 | Perceiver 1e-5, GPT-2 1e-6 | 0.0336 |
| **2c** | CrossAttention blocks only | Frozen | Switched to injection (Perceiver loaded from 2a) | 25 | 1e-5 | — |
| **2d** | All components | Unfrozen | Injection (final architecture) | 25 | 1e-5 | **0.0656** |

**Phase 2a — Prepend approach:** image latents `(B, 16, 768)` were prepended to the embedded caption, and the full sequence `(B, 16+T, 768)` was passed through frozen GPT-2. The loss was computed only on caption positions using `ignore_index=-100` on the 16 prepended image positions. This worked but created a length mismatch: GPT-2 was conditioned on image tokens only at the start, with no mechanism to re-attend to the image deeper in the network.

**Phase 2c/2d — Why switch to injection:** the prepend approach forces GPT-2 to carry image information through 12 layers of self-attention, where it progressively dilutes. Injecting cross-attention at every layer maintains a direct image-to-text channel regardless of sequence position or depth.

## 8. Final Training — Phase 2d

Phase 2d fine-tunes all components jointly using the injection architecture. Perceiver and GPT-2 weights are loaded from Phase 2b; CrossAttention weights from Phase 2c.

**Loss:** `CrossEntropyLoss(ignore_index=50256)` — padding token 50256 is excluded from the loss throughout Phase 3 training.

In [ ]:
import time

def train(dataloader, decoder_model, loss_fn, optimizer):
    decoder_model.train()
    size = len(dataloader.dataset)
    for batch, (train_image, train_caption) in enumerate(dataloader):
        train_image, train_caption = train_image.to(device), train_caption.to(device)
        input_seq = train_caption[:, :-1]    # (B, T-1)
        y         = train_caption[:, 1:]     # (B, T-1) — shifted targets
        pred      = decoder_model(train_image, input_seq)   # (B, 50257, T-1)
        loss      = loss_fn(pred, y)
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
        if batch % 100 == 0:
            current = (batch + 1) * len(train_image)
            print(f'loss: {loss.item():>7.4f}  [{current:>6d}/{size:>6d}]')


def evaluate(dataloader, decoder_model, loss_fn):
    decoder_model.eval()
    total_loss, num_batches = 0, len(dataloader)
    with torch.no_grad():
        for val_image, val_caption in dataloader:
            val_image, val_caption = val_image.to(device), val_caption.to(device)
            input_seq = val_caption[:, :-1]
            y         = val_caption[:, 1:]
            pred      = decoder_model(val_image, input_seq)
            total_loss += loss_fn(pred, y).item()
    avg_loss = total_loss / num_batches
    print(f'Val loss: {avg_loss:.4f}')
    return avg_loss

In [ ]:
EPOCHS   = 25
LR       = 1e-5
PATIENCE = 5

decoder_model = PercieverGptDecoder().to(device)
decoder_model = nn.DataParallel(decoder_model)  # multi-GPU

# Phase 2d: load Perceiver + GPT-2 weights from Phase 2b, CrossAttention weights from Phase 2c
decoder_model.module.perciever.load_state_dict(
    torch.load('/kaggle/working/phase2b_perciever.pth', weights_only=True))
decoder_model.module.gpt2.gpt2_blocks.load_state_dict(
    torch.load('/kaggle/working/phase2b_gpt2.pth', weights_only=True))
decoder_model.module.gpt2.cross_attn.load_state_dict(
    torch.load('/kaggle/working/phase2c_cross_attention.pth', weights_only=True))

loss_fn   = nn.CrossEntropyLoss(ignore_index=50256)  # padding masked
optimizer = torch.optim.Adam(decoder_model.parameters(), lr=LR)

best_val_loss    = float('inf')
patience_counter = 0

for epoch in range(EPOCHS):
    start = time.perf_counter()
    print(f'Epoch {epoch+1}\n{"-"*50}')
    train(train_dataloader, decoder_model, loss_fn, optimizer)
    val_loss = evaluate(val_dataloader, decoder_model, loss_fn)

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(decoder_model.module.perciever.state_dict(),        '/kaggle/working/best_perciever.pth')
        torch.save(decoder_model.module.gpt2.gpt2_blocks.state_dict(), '/kaggle/working/best_gpt2.pth')
        torch.save(decoder_model.module.gpt2.cross_attn.state_dict(),  '/kaggle/working/best_cross_attention.pth')
        print('  Saved best weights.')
        patience_counter = 0
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print(f'  Early stopping at epoch {epoch+1}.')
            break

    print(f'  Time: {time.perf_counter()-start:.1f}s\n')

print('Training complete.')

## 9. Inference

Three changes from v3a:

1. **Top-k reduced: 50 → 20.** GPT-2's pretrained distribution is sharper than v3a's randomly-initialized decoder — a pool of 50 candidates includes too many incoherent tokens. Narrowing to 20 keeps sampling creative without degrading fluency.
2. **Temperature: 1.0 → 0.7.** Sharpens the softmax before multinomial sampling, making the model more decisive and reducing generation variance across runs.
3. **Retry logic.** If the first decoded token is not a known English word, or if repeated tokens are detected mid-sequence, generation is retried up to 5 times. GPT-2 occasionally starts with a BPE subword fragment (`Ġ`-prefixed) that produces grammatically broken outputs — this guard catches those cases.

In [ ]:
# Load best weights
decoder_model.module.perciever.load_state_dict(
    torch.load('/kaggle/working/best_perciever.pth', weights_only=True))
decoder_model.module.gpt2.gpt2_blocks.load_state_dict(
    torch.load('/kaggle/working/best_gpt2.pth', weights_only=True))
decoder_model.module.gpt2.cross_attn.load_state_dict(
    torch.load('/kaggle/working/best_cross_attention.pth', weights_only=True))
decoder_model.eval()

english_words = set(w.lower() for w in nltk_words.words())


def caption_generator(image_feature, retry=0):
    full_stop_token = 764     # GPT-2 token id for '.'
    T               = 0.7    # temperature
    generated       = [BOS_ID]
    repeat          = 0

    with torch.no_grad():
        image_batch = image_feature.unsqueeze(0).to(device)   # (1, 64, 768)

        for i in range(1000):
            cap_tensor          = torch.tensor(generated).unsqueeze(0).to(device)
            logits              = decoder_model(image_batch, cap_tensor)  # (1, 50257, T)
            top_vals, top_idx   = torch.topk(logits[0, :, -1], 20)
            probs               = torch.softmax(top_vals / T, dim=-1)
            choice              = torch.multinomial(probs, 1, replacement=True).item()
            next_token          = top_idx[choice].item()

            # On the first generated token, check it starts a valid English word
            if i == 0:
                first_str = tokenizer.convert_ids_to_tokens([next_token])[0].lower()
                if first_str not in english_words:
                    repeat += 1

            if next_token == EOS_ID or next_token == 0:
                break
            generated.append(next_token)
            if next_token == full_stop_token:
                break
            if len(generated) > 3 and generated[-1] == generated[-2] == generated[-3]:
                break

    # Detect degenerate output: all-caps words or in-sequence triple repeats
    sentence   = tokenizer.decode(generated[1:]).strip()
    words      = sentence.split()
    has_allcaps  = sum(1 for w in words if w.isupper() and len(w) > 3) > 0
    has_trirepeat = (len(words) >= 3 and
                     any(words[i] == words[i+1] == words[i+2] for i in range(len(words)-2)))

    if (has_allcaps or repeat > 0 or has_trirepeat) and retry < 5:
        return caption_generator(image_feature, retry + 1)

    return sentence


# Qualitative check — 5 random validation images
for key in random.sample(val_keys, 5):
    feat  = torch.from_numpy(get_embedding(key).copy()).float()
    pred  = caption_generator(feat)
    truth = list(flickr_captions_dict[key].values())[0]
    print(f'Image: {key}')
    print(f'True:  {truth}')
    print(f'Pred:  {pred}')
    print()

## 10. BLEU-4 Evaluation

Multi-reference BLEU-4 over the full Flickr30k validation set (3,179 images, 5 references each). Same protocol as v3a.

In [ ]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^\w\s]', '', text)
    return text.strip()


smoothie    = SmoothingFunction().method4
bleu_scores = []

for key in val_keys:
    feat       = torch.from_numpy(get_embedding(key).copy()).float()
    predicted  = caption_generator(feat)
    references = [clean_text(cap).split() for cap in flickr_captions_dict[key].values()]
    hypothesis = clean_text(predicted).split()
    score      = sentence_bleu(references, hypothesis,
                               smoothing_function=smoothie,
                               weights=(0.25, 0.25, 0.25, 0.25))
    bleu_scores.append(score)

avg_bleu = sum(bleu_scores) / len(bleu_scores)
print(f'Average BLEU-4 on {len(val_keys)} validation images: {avg_bleu:.4f}')
# OUT: Average BLEU Score: 0.0656

## Results

**BLEU-4: 0.0656** — marginally below v3a (0.0674), despite a significantly more complex architecture.

This is initially surprising. The explanation:

- **GPT-2 fine-tuning on Flickr30k alone is insufficient.** Flickr30k has only 28,604 training images. Fine-tuning a 124M-parameter language model on ~143K caption pairs does not provide enough signal to substantially shift GPT-2's weights without forgetting.
- **The CrossAttention blocks are newly initialized** and have to learn from scratch, competing with well-calibrated GPT-2 weights. The signal-to-noise ratio is low in early training.
- **Flickr30k captions are short and formulaic.** The dataset reward is similar across architectures — both v3a and v3b generate plausible descriptions, so BLEU scores are close regardless of model complexity.

The gap opens on COCO 2017 in v3c: with 414,113 training captions and more visual diversity, the Perceiver + CrossAttention architecture has enough data to fully exploit its design advantages, reaching BLEU **0.1341**.

Sample predictions:

| True caption | v3a predicted | v3b predicted |
|---|---|---|
| A man in a red shirt climbing a rock face | A man is standing on a mountain | A man is climbing up a steep rock |
| Children playing in a fountain on a hot day | A group of people are walking in a park | Children are playing in the water |
| A dog catches a frisbee in mid-air | A dog is running in a field | A dog is jumping to catch a ball |

## What This Version Revealed

### What worked
- **CrossAttention injection at every GPT-2 layer:** qualitative outputs are visibly more specific than v3a — the model correctly identifies actions and objects it missed before ("climbing" vs "standing on", "jumping" vs "running").
- **Retry logic:** eliminates the most obvious degenerate outputs (BPE fragments, all-caps runs, triple repeats) without affecting the majority of valid generations.
- **`ignore_index=50256`:** loss curves are cleaner and val loss drops further before plateauing — padding no longer inflates apparent model confidence.
- **Progressive training showed measurable signal at each phase.** Phase 2b (prepend, Perceiver + GPT-2 jointly fine-tuned) reached BLEU **0.0336** on the Flickr val set — confirming that image conditioning was effective even before the switch to cross-attention injection. Phase 2d raised this to **0.0656**.

### What didn't work — and why v3c changed it

1. **Flickr30k is too small for this architecture.** A Perceiver + GPT-2 + 12×CrossAttention system has far more parameters than v3a's custom Transformer. On 28K images it overfits quickly despite DataParallel and gradient checkpointing.

2. **Staged unfreezing caused weight conflicts.** Loading Phase 2a Perceiver weights into the Phase 2c/2d architecture (which has different forward semantics — injection vs prepend) introduced subtle inconsistencies in how the latents were used.

3. **BLEU on Flickr30k has a low ceiling.** The dataset's captions are short (avg 11 tokens), so BLEU-4 scores are inherently compressed — the difference between a good model and a great one can be <0.01 BLEU. COCO 2017's 414K captions and 80 object categories provide a richer evaluation surface.

## Architecture Comparison Table

| Version | Notebook | Framework | Dataset | Image Encoder | Decoder | BLEU-4 |
|---|---|---|---|---|---|---|
| v0 (Colab) | — | TF/Keras | Instagram | InceptionV3 `layers[-2]` flat 2048 | Embedding + LSTM(256) | ~0 (crash) |
| v1 | `v1_inceptionv3_lstm` | TF/Keras | Instagram | InceptionV3 `layers[-2]` flat 2048 | Embedding + LSTM(512), Add/Concat | ~0 (collapse) |
| v2 | `v2_inceptionv3_attention` | TF/Keras | Instagram | InceptionV3 `mixed10` spatial (64×2048) | Embedding + TemporalImageAttention + LSTM(256) | 0.026 |
| v3a | `v3a_convnext_transformer` | PyTorch | Flickr30k | ConvNeXt-Tiny multi-scale (64×768) | ProjectionHead + Custom TransformerDecoder (6L, 8H, 512) | 0.0674 |
| **v3b** (this) | `v3b_perceiver_gpt2` | PyTorch | Flickr30k | ConvNeXt-Tiny multi-scale (64×768) | Perceiver(16 latents) + GPT-2 + CrossAttention(×12) | **0.0656** |
| v3c | `v3c_coco_finetuning` | PyTorch | COCO 2017 | ConvNeXt-Tiny multi-scale (64×768) | Perceiver + GPT-2 + CrossAttention(×12), COCO fine-tuned | **0.1341** |